In [ ]:
%pip install timm==0.9.12 torch torchmetrics torchvision --index-url https://download.pytorch.org/whl/cu121 --upgrade tqdm opencv-python pillow --upgrade

In [ ]:
import torch
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0)) 

In [ ]:
# Cell 1 - Imports and Setup
from pathlib import Path
import hashlib, cv2, random, time
import numpy as np
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from tqdm import tqdm
from collections import Counter, defaultdict
from torch.utils.data import Dataset, DataLoader
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, 
    roc_curve, 
    auc, 
    precision_recall_curve,
    average_precision_score,
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Paths
DATA_ROOT = Path(r"G:/My Drive/CLPD-MF-Dataset")
MODEL_PATH = Path("C:/Users/Mohamed Hazem/Graduation Project/Dr. Rushdy/CLPD Dr. Kariman/Mycosis-Fungoides-Classifier/Trained Models")
OUTPUT_PATH = Path("./evaluation_results")
OUTPUT_PATH.mkdir(exist_ok=True)

assert DATA_ROOT.exists(), f"Dataset folder not found at {DATA_ROOT}"
assert MODEL_PATH.exists(), f"Model folder not found at {MODEL_PATH}"

print(f"Dataset root: {DATA_ROOT}")
print(f"Model path: {MODEL_PATH}")
print(f"Output path: {OUTPUT_PATH}")

In [ ]:
# Cell 2 - Data Loading Functions (Reuse from Training)
def list_images_and_labels(root):
    """List all images with labels and magnifications"""
    rows = []
    root = Path(root)
    
    # Process MF folder
    mf_dir = root / "MF"
    if mf_dir.exists():
        for patient_dir in mf_dir.iterdir():
            if not patient_dir.is_dir(): continue
            patient_name = patient_dir.name
            for mag in ['x10', 'x20']:
                mag_dir = patient_dir / mag
                if mag_dir.exists() and mag_dir.is_dir():
                    for img_path in mag_dir.glob('*.tif'):
                        rows.append({
                            'path': img_path,
                            'label': 'MF',
                            'patient': patient_name,
                            'mag': mag,
                            'subtype': None
                        })
    
    # Process Non-MF folder
    nonmf_dir = root / "Non-MF"
    if nonmf_dir.exists():
        for subtype_dir in nonmf_dir.iterdir():
            if not subtype_dir.is_dir(): continue
            subtype = subtype_dir.name
            for patient_dir in subtype_dir.iterdir():
                if not patient_dir.is_dir(): continue
                patient_name = patient_dir.name
                for mag in ['x10', 'x20']:
                    mag_dir = patient_dir / mag
                    if mag_dir.exists() and mag_dir.is_dir():
                        for img_path in mag_dir.glob('*.tif'):
                            rows.append({
                                'path': img_path,
                                'label': 'Non-MF',
                                'patient': patient_name,
                                'mag': mag,
                                'subtype': subtype
                            })
    return rows

print("\nLoading dataset...")
all_images = list_images_and_labels(DATA_ROOT)
print(f"Total images found: {len(all_images)}")

In [ ]:
# Cell 3 - Patch Extraction and Dataset (Reuse from Training)
PATCH_CACHE = Path('./patch_cache')
PATCH_CACHE.mkdir(exist_ok=True)

def extract_and_cache_patches(img_path, patch_size=512, stride=256, 
                              min_foreground_ratio=0.285, max_patches_per_image=200):
    key = hashlib.sha1(str(img_path).encode()).hexdigest()
    cache_dir = PATCH_CACHE / key
    if cache_dir.exists() and any(cache_dir.iterdir()):
        return sorted([str(p) for p in cache_dir.glob('*.jpg')])

    cache_dir.mkdir(parents=True, exist_ok=True)
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    patches = []

    for y in range(0, H-patch_size+1, stride):
        for x in range(0, W-patch_size+1, stride):
            crop = img.crop((x, y, x+patch_size, y+patch_size))
            arr = np.asarray(crop)
            
            hsv_img = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
            saturation = hsv_img[:, :, 1]
            fg_ratio = (saturation > 20).mean()

            if fg_ratio < min_foreground_ratio: continue
            
            fname = cache_dir / f'{x}_{y}.jpg'
            crop.save(fname, quality=90)
            patches.append(str(fname))
            if len(patches) >= max_patches_per_image: break
        if len(patches) >= max_patches_per_image: break
    return patches

class MFHistologyDataset(Dataset):
    def __init__(self, rows, mag='x20', mode='train', patching=True, patch_size=512,
                 stride=256, transforms=None, max_patches_per_image=100):
        self.rows = [r for r in rows if (mag is None or r['mag']==mag)]
        self.mode = mode
        self.patching = patching
        self.patch_size = patch_size
        self.stride = stride
        self.max_patches_per_image = max_patches_per_image
        self.transforms = transforms
        labels = sorted(list({r['label'] for r in self.rows}))
        self.label2idx = {lab:i for i, lab in enumerate(labels)}

        self.items = []
        for r in self.rows:
            if self.patching:
                patches = extract_and_cache_patches(r['path'], patch_size=self.patch_size,
                                                    stride=self.stride, max_patches_per_image=self.max_patches_per_image)
                for p in patches:
                    self.items.append({
                        'img': p, 
                        'label': self.label2idx[r['label']], 
                        'source': str(r['path']),
                        'patient': r['patient']
                    })
            else:
                self.items.append({
                    'img': str(r['path']), 
                    'label': self.label2idx[r['label']], 
                    'source': str(r['path']),
                    'patient': r['patient']
                })

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        it = self.items[idx]
        img = Image.open(it['img']).convert('RGB')
        if self.transforms: img = self.transforms(img)
        return img, it['label'], it['source'], it['patient']

# Transforms
val_tf = transforms.Compose([
    transforms.Resize((512, 512), interpolation=InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [ ]:
# Cell 4 - Data Split (Reuse from Training)
def patient_split_stratified(rows, mag='x20', val_frac=0.15, seed=45):
    cls_map = {}
    for r in rows:
        if mag is not None and r['mag'] != mag: continue
        cls_map.setdefault(r['label'], {}).setdefault(r['patient'], []).append(r)

    train_rows, val_rows = [], []
    rng = random.Random(seed)

    for cls, patients_dict in cls_map.items():
        patients = list(patients_dict.keys())
        rng.shuffle(patients)
        n = len(patients)
        n_val = max(1, int(n * val_frac))
        val_p = set(patients[:n_val]) 

        for p, rlist in patients_dict.items():
            if p in val_p:  
                val_rows += rlist
            else: 
                train_rows += rlist
    return train_rows, val_rows

# Create three-way split
train_val_rows, test_rows = patient_split_stratified(all_images, mag=None, val_frac=0.15, seed=45)
val_frac_adjusted = 0.15 / (1 - 0.15)
train_rows, val_rows = patient_split_stratified(train_val_rows, mag=None, val_frac=val_frac_adjusted, seed=45)

print(f"\nTotal patients: {len(set(r['patient'] for r in all_images))}")
print(f"Training patients: {len(set(r['patient'] for r in train_rows))}")
print(f"Validation patients: {len(set(r['patient'] for r in val_rows))}")
print(f"Test patients: {len(set(r['patient'] for r in test_rows))}")

# Create test datasets
test_rows_10 = [r for r in test_rows if r['mag'] == 'x10']
test_rows_20 = [r for r in test_rows if r['mag'] == 'x20']

test_ds_10 = MFHistologyDataset(test_rows_10, mag='x10', mode='test', patching=True, transforms=val_tf)
test_ds_20 = MFHistologyDataset(test_rows_20, mag='x20', mode='test', patching=True, transforms=val_tf)

print(f"\nx10 Test Dataset: {len(test_ds_10)} patches")
print(f"x20 Test Dataset: {len(test_ds_20)} patches")


In [ ]:
# Cell 5 - Load Trained Models  
def create_model(model_name='resnet50', pretrained=True, num_classes=2, dropout=0.2):
    model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes, drop_rate=dropout)
    return model

# Load best hyperparameters
print("\n" + "="*80)
print("LOADING BEST HYPERPARAMETERS")
print("="*80)

with open('best_params_x10.json', 'r') as f:
    best_config_x10 = json.load(f)
    best_params_x10 = best_config_x10['best_params']
    print(f"\nx10 Best Params:")
    for k, v in best_params_x10.items():
        print(f"  {k}: {v}")

with open('best_params_x20.json', 'r') as f:
    best_config_x20 = json.load(f)
    best_params_x20 = best_config_x20['best_params']
    print(f"\nx20 Best Params:")
    for k, v in best_params_x20.items():
        print(f"  {k}: {v}")

# Load models
print("\n" + "="*80)
print("LOADING TRAINED MODELS")
print("="*80)

model_10 = create_model(
    model_name=best_params_x10['model_architecture'],
    pretrained=False,
    num_classes=2,
    dropout=best_params_x10['dropout']
)
model_10.load_state_dict(torch.load(MODEL_PATH / f"model_{best_params_x10['model_architecture']}_x10.pth"))
model_10.to(device).eval()
print(f"✓ Loaded x10 model: {best_params_x10['model_architecture']}")

model_20 = create_model(
    model_name=best_params_x20['model_architecture'],
    pretrained=False,
    num_classes=2,
    dropout=best_params_x20['dropout']
)
model_20.load_state_dict(torch.load(MODEL_PATH / f"model_{best_params_x20['model_architecture']}_x20.pth"))
model_20.to(device).eval()
print(f"✓ Loaded x20 model: {best_params_x20['model_architecture']}")

In [ ]:
# Cell 6 - Patch-Level Evaluation Function
def evaluate_patch_level(model, test_loader, device, mag):
    """Evaluate model at patch level"""
    print(f"\nEvaluating {mag} model at PATCH LEVEL...")
    
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    all_patients = []
    
    with torch.no_grad():
        for imgs, labels, sources, patients in tqdm(test_loader, desc=f"Patch-level {mag}"):
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            probs = F.softmax(outputs, dim=1)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())  # Probability of Non-MF (class 1)
            all_patients.extend(patients)
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, support = precision_recall_fscore_support(
        all_labels, all_preds, average=None, labels=[0, 1], zero_division=0
    )
    
    # Weighted metrics
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='weighted', zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # ROC and PR curves
    fpr, tpr, roc_thresholds = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    
    prec_curve, rec_curve, pr_thresholds = precision_recall_curve(all_labels, all_probs)
    pr_auc = average_precision_score(all_labels, all_probs)
    
    results = {
        'magnification': mag,
        'level': 'patch',
        'n_samples': len(all_labels),
        'accuracy': float(accuracy),
        'precision_mf': float(precision[0]),
        'recall_mf': float(recall[0]),
        'specificity_mf': float(specificity),
        'f1_mf': float(f1[0]),
        'precision_nonmf': float(precision[1]),
        'recall_nonmf': float(recall[1]),
        'specificity_nonmf': float(recall[0]),  # Specificity for Non-MF = Sensitivity for MF
        'f1_nonmf': float(f1[1]),
        'precision_weighted': float(precision_w),
        'recall_weighted': float(recall_w),
        'f1_weighted': float(f1_w),
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'confusion_matrix': cm.tolist(),
        'support': support.tolist(),
        'predictions': all_preds.tolist(),
        'labels': all_labels.tolist(),
        'probabilities': all_probs.tolist(),
        'patients': all_patients,
        'roc_curve': {
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist(),
            'thresholds': roc_thresholds.tolist()
        },
        'pr_curve': {
            'precision': prec_curve.tolist(),
            'recall': rec_curve.tolist(),
            'thresholds': pr_thresholds.tolist()
        }
    }
    
    return results

In [ ]:
# Cell 7 - Patient-Level Evaluation Function

def evaluate_patient_level(patch_results, test_rows, mag):
    """Aggregate patch predictions to patient level using mean probability"""
    print(f"\nAggregating to PATIENT LEVEL for {mag}...")
    
    # Group patches by patient
    patient_data = defaultdict(lambda: {'probs': [], 'label': None})
    
    for i, patient in enumerate(patch_results['patients']):
        prob = patch_results['probabilities'][i]
        label = patch_results['labels'][i]
        patient_data[patient]['probs'].append(prob)
        patient_data[patient]['label'] = label
    
    # Aggregate using mean
    patient_preds = []
    patient_labels = []
    patient_probs = []
    patient_names = []
    
    for patient, data in patient_data.items():
        mean_prob = np.mean(data['probs'])
        pred = 1 if mean_prob > 0.5 else 0
        
        patient_preds.append(pred)
        patient_labels.append(data['label'])
        patient_probs.append(mean_prob)
        patient_names.append(patient)
    
    patient_preds = np.array(patient_preds)
    patient_labels = np.array(patient_labels)
    patient_probs = np.array(patient_probs)
    
    # Calculate metrics
    accuracy = accuracy_score(patient_labels, patient_preds)
    precision, recall, f1, support = precision_recall_fscore_support(
        patient_labels, patient_preds, average=None, labels=[0, 1], zero_division=0
    )
    
    # Weighted metrics
    precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
        patient_labels, patient_preds, average='weighted', zero_division=0
    )
    
    # Confusion matrix
    cm = confusion_matrix(patient_labels, patient_preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # ROC and PR curves
    fpr, tpr, roc_thresholds = roc_curve(patient_labels, patient_probs)
    roc_auc = auc(fpr, tpr)
    
    prec_curve, rec_curve, pr_thresholds = precision_recall_curve(patient_labels, patient_probs)
    pr_auc = average_precision_score(patient_labels, patient_probs)
    
    results = {
        'magnification': mag,
        'level': 'patient',
        'n_patients': len(patient_labels),
        'accuracy': float(accuracy),
        'precision_mf': float(precision[0]),
        'recall_mf': float(recall[0]),
        'sensitivity_mf': float(recall[0]),  # Explicitly named
        'specificity_mf': float(specificity),
        'f1_mf': float(f1[0]),
        'precision_nonmf': float(precision[1]),
        'recall_nonmf': float(recall[1]),
        'sensitivity_nonmf': float(recall[1]),  # Explicitly named
        'specificity_nonmf': float(recall[0]),
        'f1_nonmf': float(f1[1]),
        'precision_weighted': float(precision_w),
        'recall_weighted': float(recall_w),
        'sensitivity_weighted': float(recall_w),  # Explicitly named
        'f1_weighted': float(f1_w),
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'confusion_matrix': cm.tolist(),
        'support': support.tolist(),
        'patient_predictions': patient_preds.tolist(),
        'patient_labels': patient_labels.tolist(),
        'patient_probabilities': patient_probs.tolist(),
        'patient_names': patient_names,
        'roc_curve': {
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist(),
            'thresholds': roc_thresholds.tolist()
        },
        'pr_curve': {
            'precision': prec_curve.tolist(),
            'recall': rec_curve.tolist(),
            'thresholds': pr_thresholds.tolist()
        }
    }
    
    return results

In [ ]:
# Cell 8 - Run Evaluation

print("\n" + "="*80)
print("STARTING MODEL EVALUATION")
print("="*80)

# Create dataloaders
test_loader_10 = DataLoader(test_ds_10, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)
test_loader_20 = DataLoader(test_ds_20, batch_size=16, shuffle=False, num_workers=0, pin_memory=True)

# Evaluate x10
print("\n--- Evaluating x10 Model ---")
patch_results_x10 = evaluate_patch_level(model_10, test_loader_10, device, 'x10')
patient_results_x10 = evaluate_patient_level(patch_results_x10, test_rows_10, 'x10')

# Evaluate x20
print("\n--- Evaluating x20 Model ---")
patch_results_x20 = evaluate_patch_level(model_20, test_loader_20, device, 'x20')
patient_results_x20 = evaluate_patient_level(patch_results_x20, test_rows_20, 'x20')

print("\n✓ Evaluation complete!")

In [ ]:
# Cell 9 - Print Summary

def print_metrics_summary(patch_res, patient_res):
    """Print formatted metrics summary"""
    mag = patch_res['magnification']
    
    print(f"\n{'='*80}")
    print(f"{mag.upper()} MODEL EVALUATION RESULTS")
    print(f"{'='*80}")
    
    # Patch-level
    print(f"\n┌─ PATCH-LEVEL METRICS ({patch_res['n_samples']} patches)")
    print(f"│  Accuracy: {patch_res['accuracy']:.4f}")
    print(f"│  ROC-AUC:  {patch_res['roc_auc']:.4f}")
    print(f"│  PR-AUC:   {patch_res['pr_auc']:.4f}")
    print(f"│")
    print(f"│  MF Class:")
    print(f"│    Precision:   {patch_res['precision_mf']:.4f}")
    print(f"│    Recall:      {patch_res['recall_mf']:.4f}")
    print(f"│    Specificity: {patch_res['specificity_mf']:.4f}")
    print(f"│    F1-Score:    {patch_res['f1_mf']:.4f}")
    print(f"│")
    print(f"│  Non-MF Class:")
    print(f"│    Precision:   {patch_res['precision_nonmf']:.4f}")
    print(f"│    Recall:      {patch_res['recall_nonmf']:.4f}")
    print(f"│    Specificity: {patch_res['specificity_nonmf']:.4f}")
    print(f"│    F1-Score:    {patch_res['f1_nonmf']:.4f}")
    print(f"└─")
    
    # Patient-level
    print(f"\n┌─ PATIENT-LEVEL METRICS ({patient_res['n_patients']} patients)")
    print(f"│  Accuracy: {patient_res['accuracy']:.4f}")
    print(f"│  ROC-AUC:  {patient_res['roc_auc']:.4f}")
    print(f"│  PR-AUC:   {patient_res['pr_auc']:.4f}")
    print(f"│")
    print(f"│  MF Class:")
    print(f"│    Precision:   {patient_res['precision_mf']:.4f}")
    print(f"│    ★ SENSITIVITY: {patient_res['sensitivity_mf']:.4f} ★")
    print(f"│    Specificity: {patient_res['specificity_mf']:.4f}")
    print(f"│    F1-Score:    {patient_res['f1_mf']:.4f}")
    print(f"│")
    print(f"│  Non-MF Class:")
    print(f"│    Precision:   {patient_res['precision_nonmf']:.4f}")
    print(f"│    ★ SENSITIVITY: {patient_res['sensitivity_nonmf']:.4f} ★")
    print(f"│    Specificity: {patient_res['specificity_nonmf']:.4f}")
    print(f"│    F1-Score:    {patient_res['f1_nonmf']:.4f}")
    print(f"└─")
    
    # Confusion matrices
    cm_patch = np.array(patch_res['confusion_matrix'])
    cm_patient = np.array(patient_res['confusion_matrix'])
    
    print(f"\n  Patch-Level Confusion Matrix:")
    print(f"                 Predicted MF  Predicted Non-MF")
    print(f"    Actual MF         {cm_patch[0,0]:6d}         {cm_patch[0,1]:12d}")
    print(f"    Actual Non-MF     {cm_patch[1,0]:6d}         {cm_patch[1,1]:12d}")
    
    print(f"\n  Patient-Level Confusion Matrix:")
    print(f"                 Predicted MF  Predicted Non-MF")
    print(f"    Actual MF         {cm_patient[0,0]:6d}         {cm_patient[0,1]:12d}")
    print(f"    Actual Non-MF     {cm_patient[1,0]:6d}         {cm_patient[1,1]:12d}")

print_metrics_summary(patch_results_x10, patient_results_x10)
print_metrics_summary(patch_results_x20, patient_results_x20)

In [ ]:
# Cell 10 - Save Results to JSON

print("\n" + "="*80)
print("SAVING RESULTS TO JSON")
print("="*80)

# Save patch-level results
with open(OUTPUT_PATH / 'x10_patch_metrics.json', 'w') as f:
    json.dump(patch_results_x10, f, indent=2)
print(f"✓ Saved: x10_patch_metrics.json")

with open(OUTPUT_PATH / 'x20_patch_metrics.json', 'w') as f:
    json.dump(patch_results_x20, f, indent=2)
print(f"✓ Saved: x20_patch_metrics.json")

# Save patient-level results
with open(OUTPUT_PATH / 'x10_patient_metrics.json', 'w') as f:
    json.dump(patient_results_x10, f, indent=2)
print(f"✓ Saved: x10_patient_metrics.json")

with open(OUTPUT_PATH / 'x20_patient_metrics.json', 'w') as f:
    json.dump(patient_results_x20, f, indent=2)
print(f"✓ Saved: x20_patient_metrics.json")


In [ ]:
# Cell 11 - Save Results to CSV

print("\n" + "="*80)
print("SAVING RESULTS TO CSV")
print("="*80)

# Create summary table
summary_data = []

for patch_res, patient_res in [(patch_results_x10, patient_results_x10), 
                                (patch_results_x20, patient_results_x20)]:
    mag = patch_res['magnification']
    
    # Patch-level row
    summary_data.append({
        'Magnification': mag,
        'Level': 'Patch',
        'N_Samples': patch_res['n_samples'],
        'Accuracy': patch_res['accuracy'],
        'Precision_MF': patch_res['precision_mf'],
        'Recall_MF': patch_res['recall_mf'],
        'Sensitivity_MF': patch_res['recall_mf'],
        'Specificity_MF': patch_res['specificity_mf'],
        'F1_MF': patch_res['f1_mf'],
        'Precision_NonMF': patch_res['precision_nonmf'],
        'Recall_NonMF': patch_res['recall_nonmf'],
        'Sensitivity_NonMF': patch_res['recall_nonmf'],
        'Specificity_NonMF': patch_res['specificity_nonmf'],
        'F1_NonMF': patch_res['f1_nonmf'],
        'ROC_AUC': patch_res['roc_auc'],
        'PR_AUC': patch_res['pr_auc']
    })
    
    # Patient-level row
    summary_data.append({
        'Magnification': mag,
        'Level': 'Patient',
        'N_Samples': patient_res['n_patients'],
        'Accuracy': patient_res['accuracy'],
        'Precision_MF': patient_res['precision_mf'],
        'Recall_MF': patient_res['recall_mf'],
        'Sensitivity_MF': patient_res['sensitivity_mf'],
        'Specificity_MF': patient_res['specificity_mf'],
        'F1_MF': patient_res['f1_mf'],
        'Precision_NonMF': patient_res['precision_nonmf'],
        'Recall_NonMF': patient_res['recall_nonmf'],
        'Sensitivity_NonMF': patient_res['sensitivity_nonmf'],
        'Specificity_NonMF': patient_res['specificity_nonmf'],
        'F1_NonMF': patient_res['f1_nonmf'],
        'ROC_AUC': patient_res['roc_auc'],
        'PR_AUC': patient_res['pr_auc']
    })